# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents the daily performance of one content item for one client on one report date.

Time window used in this notebook:
2026-03-01 to 2026-03-31.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features
These fields are available before the prediction time and can be used as model features.

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_pageviews

### Label
For future modeling, the target (label) can be future GSC clicks. This notebook only defines the data contract and does not create the label.

### Context
These fields are used for grouping and identification, not for model training.

- client_hash_id
- content_hash_id
- report_date

### Excluded
These fields are excluded from model features.

- client_has_gsc
- client_has_ga4
- Future information

**Reason:** Future information is excluded because it would introduce data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import duckdb

con = duckdb.connect()

local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

### Grain Verification

This query checks whether each row represents one content item for one client on one report date.

Expected result:
No rows should be returned. If the result is empty, the data grain is correct.

In [7]:
grain = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS c
FROM read_parquet('{local_file}')
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 5
""").fetch_df()

grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,c


### Row Count and Date Range

This query verifies the number of rows and confirms the time window of the dataset.

In [8]:
counts = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM read_parquet('{local_file}')
""").fetch_df()

counts

,total_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


### Data Availability

This query keeps only rows where both GSC and GA4 data are available using `IS TRUE`.

In [9]:
availability = con.execute(f"""
SELECT
    COUNT(*) AS usable_rows
FROM read_parquet('{local_file}')
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
""").fetch_df()

availability

,usable_rows
0,364347


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This analysis uses only March 2026 data.

Some clients have shorter historical data than others.

Rows where GSC or GA4 data are unavailable are excluded.

This notebook is intended for exploratory analysis and does not include future information.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.